# HW100 · 3/3 — Analysis

Per-block feasibility with confidence intervals, quality among feasible samples, how many fully feasible 120-variable assignments can be **assembled** by per-block post-selection, and the comparison with the pre-registered noise-model prediction.

**Important when reading the results.** Blocks are independent: a shot is feasible for the whole problem only if all 12 blocks are, which has probability ≈ ∏ p_b for *every* method. What per-block post-selection buys is that complete feasible assignments are assembled from accepted blocks of different shots; the yield is limited by the worst block.

In [ ]:
import os, sys, json, pickle, time, warnings; warnings.filterwarnings('ignore')
root = os.path.abspath('.')
while not os.path.isdir(os.path.join(root, 'negsearch')): root = os.path.dirname(root)      # repository root
os.chdir(root); sys.path.insert(0, root)
import numpy as np
from negsearch import hw100 as H
OUT = 'hardware_circuits/hw100'; RES = 'results/hw100'; os.makedirs(OUT, exist_ok=True); os.makedirs(RES, exist_ok=True)
print('repository root:', root)

In [ ]:
SOURCE = 'model'      # 'ibm' | 'aer' | 'model'  (folder under results/hw100/)
import pandas as pd, matplotlib; import matplotlib.pyplot as plt
meta = json.load(open(f'{OUT}/meta.json')); D = pickle.load(open(f'{OUT}/instance.pkl', 'rb')); inst, ideal = D['inst'], D['ideal']
methods = meta['methods']
bits = {m: np.load(f'{RES}/{SOURCE}/{m}.npz')['bits'] for m in methods}
print(SOURCE, {m: b.shape for m, b in bits.items()})
if SOURCE == 'model': print('WARNING: SOURCE = model only tests this notebook (samples come from the same model as the prediction, so criterion 5 passes by construction).')

In [ ]:
R = {m: H.analyse(inst, bits[m]) for m in methods}
pred = {p['method']: p for p in meta['prediction']}
rand_mean, rand_best = H.random_feasible_cost(inst); exact = H.exact_block_cost(inst)
rows = []
for m in methods:
    r = R[m]; N = r['shots']; k = sum(x['n_acc'] for x in r['blocks']); lo, hi = H.wilson(k, N * 12)
    rows.append(dict(method=m, feasible_per_block=r['block_feas_mean'], CI95=f'[{lo:.3f}, {hi:.3f}]', model_prediction=pred[m]['feasible_per_block'],
                     feasible_blocks_per_shot=r['mean_feasible_blocks'], quality=np.nanmean([x['quality'] for x in r['blocks']]),
                     assembled_solutions=r['assembled'], assembled_cost_mean=r['assembled_cost_mean'], global_feasible_shots=r['global_feasible_fraction']))
tab = pd.DataFrame(rows); tab.round(4)

In [ ]:
print(f"reference costs of the 120-variable objective: random feasible mean {rand_mean:.2f} (best of 2000: {rand_best:.2f}); exact block-coordinate optimum {exact:.2f}")
print("assembled_solutions = number of complete feasible assignments that can be built from accepted blocks = min over blocks of accepted samples")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(12); w = 0.11
for i, m in enumerate(methods):
    ax[0].bar(x + (i - 3) * w, [b['feas'] for b in R[m]['blocks']], w, label=m)
ax[0].set_xlabel('block'); ax[0].set_ylabel('measured feasible fraction'); ax[0].set_title(f'feasibility per block ({SOURCE})'); ax[0].legend(fontsize=7, ncol=2)
for m in methods:
    ax[1].scatter([s['esp'] for s in meta['compile'][m]], [b['feas'] for b in R[m]['blocks']], label=m, s=18)
ax[1].set_xlabel('block ESP (from compiled gates)'); ax[1].set_ylabel('measured feasible fraction'); ax[1].set_title('feasibility vs ESP'); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{RES}/{SOURCE}_feasibility.png', dpi=140); plt.show()

## Pre-stated success criteria
Defined before the run. Each line prints PASS/FAIL from the data; none is tuned afterwards.

1. **Feasibility above chance.** `ours_p0` has per-block feasible fraction above the 95 % upper bound of `uniform`.
2. **Feasibility against the quantum baselines.** `ours_p0` and `ours` have per-block feasibility above the upper bound of `lcu`, `unbalanced` and `xy`.
3. **Scale.** All 12 blocks (120 qubits) return at least 50 accepted samples for `ours_p0`, i.e. a complete feasible assembly exists.
4. **Quality.** `ours` has quality among feasible samples above the `ours_p0` value (the objective layer helps despite noise). Comparison with `lcu` is reported, not required.
4b. **Cost-informed preparation (reported).** `ours_tilt_p0` and `ours_tilt` use the classical mean-field tilt realised by Dicke angles only; `lcu_b`, `unbalanced_b` and `xy_b` receive the same information as the start of their product state. 2c requires `ours_tilt_p0` to have the highest feasibility; 4b compares quality.
5. **Model.** Measured feasibility of `ours_p0` lies within 0.1 (absolute) of the model prediction.

In [ ]:
def feas_ci(m):
    r = R[m]; k = sum(x['n_acc'] for x in r['blocks']); return H.wilson(k, r['shots'] * 12)
q = lambda m: np.nanmean([x['quality'] for x in R[m]['blocks']])
f = lambda m: R[m]['block_feas_mean']
checks = [
 ('1 feasibility above uniform control', f('ours_p0') > feas_ci('uniform')[1]),
 ('2a ours_p0 above all three baselines', f('ours_p0') > max(feas_ci('lcu')[1], feas_ci('unbalanced')[1], feas_ci('xy')[1])),
 ('2b ours (p=1) above all three baselines', f('ours') > max(feas_ci('lcu')[1], feas_ci('unbalanced')[1], feas_ci('xy')[1])),
 ('3 complete assemblies at 120 qubits (>= 50 per block)', R['ours_p0']['assembled'] >= 50),
 ('2c ours_tilt_p0 above the three biased baselines (same classical information)', f('ours_tilt_p0') > max(feas_ci('lcu_b')[1], feas_ci('unbalanced_b')[1], feas_ci('xy_b')[1])),
 ('4 quality: ours > ours_p0', q('ours') > q('ours_p0')),
 ('4b quality: ours_tilt_p0 vs best biased baseline (reported)', q('ours_tilt_p0') > max(q('lcu_b'), q('unbalanced_b'), q('xy_b'))),
 ('5 model within 0.1 of measured (ours_p0)', abs(f('ours_p0') - pred['ours_p0']['feasible_per_block']) < 0.1)]
for name, ok in checks: print('PASS' if ok else 'FAIL', '-', name)
print(f"\nquality among feasible samples: " + ', '.join(f'{m} {q(m):.3f}' for m in methods))

## How to read the outcome
* If 1–3 pass: feasible-by-construction states are produced at ~100 qubits on the chip, per block, with feasibility above both the unconstrained control and the warm-started baselines, which were given the favourable block decomposition. The model predicts that 2a passes and 2b (the p=1 layer) fails against `unbalanced`.
* Criterion 4 can fail: the objective layer costs 2–4× the gates of the preparation and may not pay off at these depths. That is a result, not a bug; `ours_full` shows the other end of the trade-off.
* `lcu` and `xy` are expected to be competitive in *quality among their feasible samples* (they have fewer gates), and `unbalanced` in how many feasible samples it produces once it has the warm start; the preparation alone is the clear winner on feasibility.
* What this does **not** show: a quantum advantage over classical solvers (blocks of 10 qubits are trivial classically), or optimisation quality inside blocks beyond the criteria above. It shows scale and feasibility.